# Explainable AI (XAI) Analysis
This notebook explores **Grad-CAM** and **Saliency Maps** to understand what features our CNN models focus on when making predictions.

In [ ]:
import sys
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2

# Add project root to path
sys.path.append(os.path.abspath('..'))

from src.data_loader import load_mnist_data, normalize_images, reshape_for_cnn
from src.explainability import grad_cam, saliency_map, get_overlayed_image

## Setup and Model Loading

In [ ]:
# Load the Advanced CNN
model_path = '../models/saved_models/cnn_advanced_augmented_best.keras'
model = tf.keras.models.load_model(model_path)

# Load data
(_, _), (X_test, y_test) = load_mnist_data()
X_test_norm = reshape_for_cnn(normalize_images(X_test))

## Demonstrating XAI Techniques

In [ ]:
# Select a sample digit (e.g., a "3")
sample_idx = 100 
sample_img = X_test_norm[sample_idx]
true_label = y_test[sample_idx]

# Predictions
pred_probs = model.predict(np.expand_dims(sample_img, axis=0), verbose=0)
pred_label = np.argmax(pred_probs)

# Compute Grad-CAM
heatmap = grad_cam(model, sample_img[..., 0])
overlay = get_overlayed_image(sample_img[..., 0], heatmap)

# Compute Saliency Map
saliency = saliency_map(model, sample_img)

# Visualize
plt.figure(figsize=(15, 5))

plt.subplot(1, 4, 1)
plt.imshow(sample_img[..., 0], cmap='gray')
plt.title(f"Original (True: {true_label})")
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(heatmap, cmap='jet')
plt.title("Grad-CAM Heatmap")
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(overlay)
plt.title(f"Grad-CAM Overlay (Pred: {pred_label})")
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(saliency, cmap='hot')
plt.title("Saliency Map")
plt.axis('off')

plt.tight_layout()
plt.show()